In [5]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
import os
import numpy as np

data = "data1"
if data=="data1":
    text = "group"
else:
    text = "random"

# 定义文件夹路径和CSV文件列表
data_folder = os.path.join('Data', data)
files = ['50_12_0.csv','50_12_2.csv','50_12_20.csv','50_12_TS.csv','50_12_Uni.csv']
curve_names = ['β = 0', 'β = 2', 'β = 20', 'TS', 'Uniform']
colors = px.colors.qualitative.Plotly[:len(curve_names)] 

# 创建三个DataFrame分别存储原始均值、最优均值和累积遗憾
all_mean_data = pd.DataFrame()
all_best_data = pd.DataFrame()
all_cumsum_data = pd.DataFrame()  # 新增：存储Cumulative Regret

# 循环处理每个文件
for file, name in zip(files, curve_names):
    file_path = os.path.join(data_folder, file)
    df = pd.read_csv(file_path, index_col='Repetition')
    
    # 计算每个step的均值（Instant Regret）
    mean_results = df.mean(axis=0)
    
    # 计算当前最优均值（Simple Regret，可选保留或删除）
    best_results = np.minimum.accumulate(mean_results.values)
    
    # 添加到原始均值DataFrame
    temp_mean_df = pd.DataFrame({
        'Iter': range(1, 151),
        'Value': mean_results.values,
        'Condition': name
    })
    
    # 添加到最优均值DataFrame（可选保留或删除）
    temp_best_df = pd.DataFrame({
        'Iter': range(1, 151),
        'Value': best_results,
        'Condition': name
    })    
    
    # 新增：计算Cumulative Regret（排除Uniform条件）
    if name != 'Uniform':
        cumsum_results = np.cumsum(mean_results.values)
        temp_cumsum_df = pd.DataFrame({
            'Iter': range(1, 151),
            'Value': cumsum_results,
            'Condition': name
        })
        all_cumsum_data = pd.concat([all_cumsum_data, temp_cumsum_df])
    
    all_mean_data = pd.concat([all_mean_data, temp_mean_df])
    all_best_data = pd.concat([all_best_data, temp_best_df])

# 创建子图（2行1列，垂直排列）
fig = make_subplots(
    rows=2, cols=1,
    vertical_spacing=0.1,
    subplot_titles=("Instant Regret", "Cumulative Regret")  # 添加子图标题
)

# 为每个条件添加原始均值曲线（Instant Regret，第一个子图）
for i, condition in enumerate(curve_names):
    mean_df = all_mean_data[all_mean_data['Condition'] == condition]
    fig.add_trace(
        go.Scatter(
            x=mean_df['Iter'],
            y=mean_df['Value'],
            name=condition,
            legendgroup=condition,
            showlegend=True,
            line=dict(color=colors[i]),
            meta=[condition],
            hovertemplate="Iter: %{x}<br>Mean: %{y:.4f}<br>Condition: %{meta[0]}<extra></extra>"
        ),
        row=1, col=1
    )

# 为每个条件（排除Uniform）添加累积遗憾曲线（Cumulative Regret，第二个子图）
for i, condition in enumerate(curve_names):
    if condition == 'Uniform':
        continue  # 跳过Uniform条件
    cumsum_df = all_cumsum_data[all_cumsum_data['Condition'] == condition]
    fig.add_trace(
        go.Scatter(
            x=cumsum_df['Iter'],
            y=cumsum_df['Value'],
            name=condition,
            legendgroup=condition,
            showlegend=False,  # 避免重复图例
            line=dict(color=colors[i]),
            meta=[condition],
            hovertemplate="Iter: %{x}<br>CumSum: %{y:.4f}<br>Condition: %{meta[0]}<extra></extra>"
        ),
        row=2, col=1
    )

# 更新布局
fig.update_layout(
    title_text=text+"_50_12",
    title_x=0.5,
    height=1080,
    width=1920,
    template='plotly_white',
    legend_title_text='Condition',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.95)
)

# 更新Y轴和X轴标签
fig.update_yaxes(title_text="Instant Regret", row=1, col=1)
fig.update_yaxes(title_text="Cumulative Regret", row=2, col=1)
fig.update_xaxes(title_text="Iteration", row=2, col=1)  # 仅底部子图显示X轴标签

# 显示图形
fig.show()

# 保存为HTML文件
pio.write_html(fig, '50_12_instant.html')